# The Anatomy of AI · Lecture 5 · Pictures as Numbers — the notebook

In the lecture a handwritten 3 became a list of 784 numbers, ten neurons each gave it a score, and the "3" neuron's
784 weights turned out to be a picture. Here you build all of it yourself, from scratch:

1. load the 70,000 MNIST digits and look at one as numbers;
2. write the ten neurons: multiply and add, 784 times;
3. turn scores into chances (softmax) and measure the error;
4. teach the weights with Lecture 2's rule, step by step, and watch the test score climb from 9.8% to 92.1%;
5. read the weights as pictures, and find what they are *not* (the average digit);
6. find where the model fails, and why a digit moved two squares fools it.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab, CPU only (no GPU needed), in about a minute. It downloads MNIST (11 MB) and nothing else.

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs, quiet. Free Colab already has torch; a machine without it gets the CPU-only build.
import importlib.util, subprocess, sys
def pip(*args):                                   # pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", "a GPU" if DEV == "cuda" else "the CPU")
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### The lecture's own numbers
Every number below comes from the lecture's logs (its training run, `mnist_linear.py`, and the extra measurements).
The checks compare what you compute against these.

In [ ]:
LECTURE = json.loads(r'''{"n_train": 60000, "n_test": 10000, "first_ten_of_each": [3, 10, 13, 25, 28, 55, 69, 71, 101, 126, 2, 5, 14, 29, 31, 37, 39, 40, 46, 57, 1, 35, 38, 43, 47, 72, 77, 82, 106, 119, 18, 30, 32, 44, 51, 63, 68, 76, 87, 90, 4, 6, 19, 24, 27, 33, 42, 48, 49, 56, 8, 15, 23, 45, 52, 53, 59, 102, 120, 127, 11, 21, 22, 50, 54, 66, 81, 88, 91, 98, 0, 17, 26, 34, 36, 41, 60, 64, 70, 75, 61, 84, 110, 128, 134, 146, 177, 179, 181, 184, 7, 9, 12, 16, 20, 58, 62, 73, 78, 92], "digit30": {"label": 3, "inked": 136, "full": 43, "lens_rows_15_20_cols_14_19": [[0.2, 0.51, 0.84, 0.99, 0.99, 0.75], [0.0, 0.0, 0.08, 0.64, 1.0, 0.99], [0.0, 0.0, 0.08, 0.8, 0.99, 0.99], [0.0, 0.0, 0.52, 0.99, 1.0, 0.36], [0.08, 0.56, 0.99, 0.99, 0.91, 0.12], [0.84, 0.99, 1.0, 0.84, 0.0, 0.0]], "scores": [-3.836, -3.577, -2.948, 9.072, -4.868, 3.721, -4.456, 3.51, 0.604, 2.778], "parts": {"3": [10.27, -0.91, -0.29, 9.07], "5": [6.6, -4.25, 1.37, 3.72], "8": [6.4, -4.3, -1.5, 0.6], "2": [4.07, -7.13, 0.11, -2.95]}, "probs": [0.0, 0.0, 0.0, 0.9895, 0.0, 0.0047, 0.0, 0.0038, 0.0002, 0.0018]}, "n_numbers": 7850, "curve": {"0": [0.098, 2.3026], "1": [0.1563, 2.214], "2": [0.5066, 2.1096], "5": [0.6997, 1.8616], "10": [0.733, 1.5438], "20": [0.7829, 1.1749], "50": [0.8529, 0.7764], "100": [0.8698, 0.5893], "200": [0.8867, 0.4683], "500": [0.9008, 0.3747], "1000": [0.9086, 0.3322], "2000": [0.9176, 0.3034], "6000": [0.9206, 0.2803]}, "test_acc": 0.9206, "train_acc": 0.9219, "seed_accs": [0.9216, 0.9226, 0.9221], "weights": {"min": -0.6937, "max": 0.7946, "frac_pos": 0.3511, "bias": [-0.4046, 0.3656, 0.1113, -0.2872, 0.0331, 1.3684, -0.0961, 0.6717, -1.4979, -0.2644]}, "never_inked": 67, "most_inked_square": [15, 15, 0.7261], "mean_template_acc": 0.631, "nearest_mean_acc": 0.8203, "corr_with_average": [0.224, 0.497], "corr_with_difference": [0.6, 0.777], "per_digit_acc": [0.9827, 0.978, 0.8769, 0.9129, 0.9389, 0.8688, 0.952, 0.9173, 0.8768, 0.892], "commonest_mistakes": [[2, 8, 42], [9, 4, 40], [7, 9, 34], [8, 5, 32], [4, 9, 32], [5, 3, 31]], "surest_mistake": {"index": 1790, "label": 2, "said": 9, "scores": [-9.323, -4.029, -4.949, -1.829, 4.945, -2.379, -5.522, 8.794, 5.314, 8.978]}, "shift_acc": [0.9206, 0.8655, 0.6652, 0.4469, 0.2367], "digit8570": [{"shift": 0, "pred": 3, "score_true": 14.29, "score_pred": 14.29}, {"shift": 1, "pred": 3, "score_true": 11.17, "score_pred": 11.17}, {"shift": 2, "pred": 8, "score_true": 6.57, "score_pred": 7.53}, {"shift": 3, "pred": 8, "score_true": 2.44, "score_pred": 8.17}, {"shift": 4, "pred": 8, "score_true": -0.63, "score_pred": 7.98}], "digit30_shifted_score3": [9.072, 10.074, 10.177, 9.814, 9.069], "wd_1e-3": {"acc": 0.9188, "corr": 0.998}, "random_start": {"acc": 0.9214, "corr": 0.9972}, "surest_7_as_9": {"index": 3836, "score9": 8.12, "score7": 3.07}}''')
LECTURE['curve'] = {int(k): v for k, v in LECTURE['curve'].items()}
LECTURE['digit30']['parts'] = {int(k): v for k, v in LECTURE['digit30']['parts'].items()}
print(len(json.dumps(LECTURE)), 'characters of logged numbers')

## 1. Seventy thousand digits (Plate II)
**MNIST** is 70,000 handwritten digits gathered by LeCun and colleagues (1998). Each is a small grey picture,
28 squares by 28. 60,000 are for learning (the *training set*); 10,000 are held back as a *test set* the model never
sees while it learns, so the score on them is honest.

We read the four original files from the mirror torchvision uses (no key needed). Each square holds how much ink it
has, stored as 0–255; we divide by 255, so 0 is blank paper and 1 is full ink.

In [ ]:
import gzip, requests
MIRROR = "https://ossci-datasets.s3.amazonaws.com/mnist/"

def fetch(name, offset):
    raw = requests.get(MIRROR + name, timeout=120).content
    return np.frombuffer(gzip.decompress(raw), np.uint8, offset=offset)

tr_img = fetch("train-images-idx3-ubyte.gz", 16).reshape(-1, 28, 28)
tr_lab = fetch("train-labels-idx1-ubyte.gz", 8).astype(int)
te_img = fetch("t10k-images-idx3-ubyte.gz", 16).reshape(-1, 28, 28)
te_lab = fetch("t10k-labels-idx1-ubyte.gz", 8).astype(int)
print("downloaded MNIST:", tr_img.shape[0], "training digits and", te_img.shape[0], "test digits")

xtr = tr_img.reshape(-1, 784).astype(np.float32) / 255        # one row of 784 numbers per digit
xte = te_img.reshape(-1, 784).astype(np.float32) / 255
check("training digits", len(xtr), LECTURE["n_train"])
check("test digits", len(xte), LECTURE["n_test"])

# the lecture's Plate II shows the first ten test digits of each kind
first_ten = [int(i) for c in range(10) for i in np.where(te_lab == c)[0][:10]]
check("the first ten test digits of each kind", first_ten, LECTURE["first_ten_of_each"])
INKMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("ink", [PAPER, INK])
fig, axs = plt.subplots(10, 10, figsize=(6, 6))
for ax, i in zip(axs.T.flat, first_ten):
    ax.imshow(te_img[i], cmap=INKMAP); ax.axis("off")
fig.suptitle("one hundred real test digits, ten of each", y=0.93)
plt.show()

## 2. A digit is 784 numbers (Plate III)
The lecture's running example is **test digit #30**, a 3. The computer stores it as 28 × 28 numbers. Read row after
row, that is one list of 784 numbers: the network never sees a shape, only this list.

In [ ]:
x30 = xte[30]
check("digit #30 is a", int(te_lab[30]), LECTURE["digit30"]["label"])
check("squares with any ink", int((x30 > 0).sum()), LECTURE["digit30"]["inked"])
check("squares fully black (0.99 or more)", int((x30 >= 0.99).sum()), LECTURE["digit30"]["full"])

# the lecture's lens: rows 15-20, columns 14-19 (counting from 0), to two decimals
patch = x30.reshape(28, 28)[15:21, 14:20].round(2)
print(patch)
check("the lens on Plate III: largest difference, square by square", np.abs(patch - np.array(LECTURE["digit30"]["lens_rows_15_20_cols_14_19"])).max(), 0, tol=0.005)

fig, axs = plt.subplots(1, 2, figsize=(8, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
axs[0].imshow(x30.reshape(28, 28), cmap=INKMAP); axs[0].set_title("test digit #30"); axs[0].axis("off")
axs[0].add_patch(plt.Rectangle((13.5, 14.5), 6, 6, fill=False, ec=INK, lw=1.5))
axs[1].axis("off"); axs[1].set_title("the lens, as numbers")
for r in range(6):
    for c in range(6):
        axs[1].text(c, -r, f"{patch[r, c]:.2f}", ha="center", va="center", fontsize=10,
                    color=INK if patch[r, c] > 0 else INK2)
axs[1].set_xlim(-0.6, 5.6); axs[1].set_ylim(-5.6, 0.6)
plt.show()

## 3. Ten neurons (Plates IV and VI)
One neuron per digit. The "3" neuron has its own **weight** for each of the 784 squares, and a **bias**, exactly like
Lecture 1's neuron with two inputs, only longer. Its **score** for a picture is Lecture 1's sum:

$$\text{score}_3 = w_3[0]\,x[0] + w_3[1]\,x[1] + \dots + w_3[783]\,x[783] + b_3$$

All ten neurons together: a 10 × 784 table of weights `W` and ten biases `b`. These neurons have no gate (no ReLU):
we compare their scores, negative ones too. The **answer** is the digit whose neuron scores highest.

In [ ]:
def scores(W, b, x):
    """Every neuron's score for every picture: multiply and add, 784 times. x: (n, 784) -> (n, 10)."""
    return x @ W.T + b

W0, b0 = np.zeros((10, 784), np.float32), np.zeros(10, np.float32)
check("numbers to learn: 10 x 784 weights + 10 biases", W0.size + b0.size, LECTURE["n_numbers"])

## 4. Learning the weights (Plates IX and X, used now, opened up in §8)
The weights we will read in Plates V–VIII were learned. Here is the whole recipe, from scratch; §8 takes it apart.

- **Scores to chances (softmax):** raise e to each score and share out 100% in proportion.
- **Error:** −log(the chance given to the right digit). Sure and right: near 0. At 10% each: −log 0.1 = 2.303.
- **One step downhill (Lecture 2's rule):** each weight moves by −learning rate × (its pixel) × (chance − 1 if this is
  the right digit's neuron, else chance), averaged over a *batch* of 100 digits.

The lecture's recipe: every weight starts at 0; batches of 100; learning rate 0.1; 10 passes over the 60,000 training
digits, so 6,000 steps. To meet the lecture's numbers exactly we also take the digits in the lecture's order: it
shuffled them with PyTorch's random generator, seeded 0, so we borrow that one function (`torch.randperm`) and do
everything else in NumPy.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max(1, keepdims=True))          # subtracting the biggest score first changes nothing, avoids overflow
    return e / e.sum(1, keepdims=True)


def error(W, b, x, y):
    p = softmax(scores(W, b, x))
    return float(-np.log(p[np.arange(len(y)), y]).mean())


def accuracy(W, b, x, y):
    return float((scores(W, b, x).argmax(1) == y).mean())


def train(seed=0, lr=0.1, passes=10, decay=0.0, start=None, record=()):
    """The lecture's training: SGD on softmax error, batches of 100. Returns weights, biases and a test record."""
    g = torch.Generator().manual_seed(seed)
    W = np.zeros((10, 784), np.float32) if start is None else start(g)
    b = np.zeros(10, np.float32)
    seen, step = {}, 0
    if 0 in record:
        seen[0] = (accuracy(W, b, xte, te_lab), error(W, b, xte, te_lab), W.copy())
    for _ in range(passes):
        order = torch.randperm(len(xtr), generator=g).numpy()
        for i in range(0, len(xtr), 100):
            idx = order[i:i + 100]
            x, y = xtr[idx], tr_lab[idx]
            p = softmax(scores(W, b, x))
            p[np.arange(len(y)), y] -= 1                 # chance minus 1 for the right digit: how wrong, and which way
            W -= lr * (p.T @ x / len(y) + decay * W)     # each weight moves by its pixel times the error
            b -= lr * (p.mean(0) + decay * b)
            step += 1
            if step in record:
                seen[step] = (accuracy(W, b, xte, te_lab), error(W, b, xte, te_lab), W.copy())
    return W, b, seen


t = time.time()
W, b, curve = train(record=set(LECTURE["curve"]))
print(f"6,000 steps in {time.time() - t:.1f} s")
check("test accuracy after 6,000 steps", accuracy(W, b, xte, te_lab), LECTURE["test_acc"], tol=0.0005)
check("training accuracy", accuracy(W, b, xtr, tr_lab), LECTURE["train_acc"], tol=0.0005)

## 5. Weights are a picture (Plate V)
Draw each of the "3" neuron's 784 weights on the square it belongs to. The lecture's convention: a **positive** weight
is ink, darker the bigger it is (ink there counts *for* a 3); a **negative** weight is a hollow ring, bigger the more
negative (ink there counts *against*). A 3 appears. Lay digit #30 over it: most strokes land on dark squares.

In [ ]:
def template(ax, w, title="", over=None):
    """A neuron's 784 weights in the lecture's style: ink for positive, hollow rings for negative."""
    w = w.reshape(28, 28)
    top = max(w.max(), -w.min())
    ax.imshow(np.clip(w, 0, None) / top, cmap=INKMAP, vmin=0, vmax=1)
    r, c = np.where(w < 0)
    ax.scatter(c, r, s=20 * (-w[r, c] / top), facecolors="none", edgecolors=INK, linewidths=0.5)
    if over is not None:
        ax.contour(over.reshape(28, 28), levels=[0.5], colors=[INK2], linewidths=1.2)
    ax.set_title(title, fontsize=10); ax.axis("off")


fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.8))
template(axs[0], W[3], 'the "3" neuron\'s weights')
template(axs[1], W[3], "with digit #30's strokes over it", over=x30)
plt.show()
on_dark = float((W[3][x30 > 0.5] > 0).mean())
print(f"{on_dark:.0%} of digit #30's inked squares (ink over 0.5) sit on positive weights of the 3 neuron")
check("weights range from", float(W.min()), LECTURE["weights"]["min"], tol=0.01)
check("weights range to", float(W.max()), LECTURE["weights"]["max"], tol=0.01)
check("share of weights that are positive", float((W > 0).mean()), LECTURE["weights"]["frac_pos"], tol=0.003)

## 6. Multiply and add, 784 times (Plates VI and VII)
Split the "3" neuron's score for digit #30 into its parts: ink on positive weights adds, ink on negative weights takes
away, then the bias. Then let all ten neurons score the same digit: the biggest score is the answer.

In [ ]:
prod = W * x30                                                # (10, 784): weight times pixel, square by square
s30 = scores(W, b, x30[None])[0]
for d in (3, 5, 8, 2):
    pos, neg = prod[d][prod[d] > 0].sum(), prod[d][prod[d] < 0].sum()
    print(f'"{d}" neuron: {pos:+.2f} from ink on positive weights, {neg:+.2f} on negative, bias {b[d]:+.2f}, '
          f"score {s30[d]:+.2f}")
    want = LECTURE["digit30"]["parts"][d]
    check(f'"{d}": ink on positive weights', pos, want[0], tol=0.02)
    check(f'"{d}": ink on negative weights', neg, want[1], tol=0.02)
    check(f'"{d}": bias', b[d], want[2], tol=0.01)
for d in range(10):
    check(f'score of "{d}" for digit #30', s30[d], LECTURE["digit30"]["scores"][d], tol=0.02)
check("the answer (biggest score)", int(s30.argmax()), 3)
check("the runner-up", int(np.argsort(-s30)[1]), 5)

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.bar(range(10), s30, color=[INK if d == 3 else PAPER for d in range(10)], edgecolor=INK, hatch=None)
ax.axhline(0, color=INK, lw=0.8); ax.set_xticks(range(10)); ax.set_xlabel("neuron (digit)"); ax.set_ylabel("score")
ax.set_title("ten neurons, ten scores for digit #30")
plt.show()

## 7. From scores to chances (Plate VIII)
Raise e (2.718…) to each score: big scores become huge, negative ones tiny. Share out 100% in proportion. This is
**softmax**.

In [ ]:
p30 = softmax(s30[None])[0]
print("chances, digits 0-9:", np.round(p30, 4))
for d in (3, 5, 7):
    check(f"P({d}) for digit #30", p30[d], LECTURE["digit30"]["probs"][d], tol=0.0005)
check("the error on digit #30: -log P(3)", -np.log(p30[3]), 0.0106, tol=0.0005)

## 8. How the weights got there (Plates IX and X)
At step 0 every weight is 0, so every score is 0 and every chance 10%: the error is −log 0.1 = 2.303, and the tie goes
to the first digit, "0", so the accuracy is exactly the share of zeros in the test set. Then the climb.

In [ ]:
print(" step   test accuracy   (lecture)    test error   (lecture)")
for s in sorted(curve):
    a, e, _ = curve[s]
    print(f"{s:5d}   {a:8.2%}       ({LECTURE['curve'][s][0]:.2%})    {e:8.4f}    ({LECTURE['curve'][s][1]:.4f})")
check("accuracy at step 0 = the share of zeros", curve[0][0], float((te_lab == 0).mean()))
check("accuracy at step 0", curve[0][0], LECTURE["curve"][0][0], tol=1e-4)
check("error at step 0 = -log 0.1", curve[0][1], math.log(10), tol=1e-4)
check("accuracy after 2 steps", curve[2][0], LECTURE["curve"][2][0], tol=0.002)
check("accuracy after 100 steps", curve[100][0], LECTURE["curve"][100][0], tol=0.002)
check("error after 6,000 steps", curve[6000][1], LECTURE["curve"][6000][1], tol=0.001)

fig, axs = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1.4, 2]})
st = sorted(curve)
axs[0].plot([max(s, 0.5) for s in st], [curve[s][0] for s in st], "o-", color=INK, ms=4)
axs[0].set_xscale("log"); axs[0].set_xlabel("step (log scale)"); axs[0].set_ylabel("test accuracy")
axs[0].set_title("the climb")
axs[1].axis("off")
for k, s in enumerate((1, 10, 100, 6000)):
    a = fig.add_axes([0.45 + k * 0.135, 0.12, 0.12, 0.7])
    template(a, curve[s][2][3], f"step {s}")
fig.suptitle('the "3" template sharpening', x=0.72, y=0.98, fontsize=10)
plt.show()

**The error that teaches.** A weight moves by (its pixel) × (how wrong the chance was). Sure, right answers barely move
anything; and a square with no ink moves nothing at all. Look at one step on digit #30 alone:

In [ ]:
p = softmax(scores(W, b, x30[None]))[0]
push = np.outer(p - np.eye(10)[3], x30)          # the downhill direction for each weight, from this one digit
print(f"P(3) = {p[3]:.4f}: the biggest push on any weight is {np.abs(push).max():.4f}")
print("blank squares of #30 get a push of exactly", np.abs(push[:, x30 == 0]).max())
check("blank squares are never pushed", float(np.abs(push[:, x30 == 0]).max()), 0.0)

## 9. The corners never learn (Plate XI)
How often is each square inked, over all 60,000 training digits? Some squares, mostly corners, never are. Their pixel
is always 0, so by the rule above their weights never move from 0.

In [ ]:
freq = (xtr > 0).mean(0)
never = freq == 0
check("squares never inked by any training digit", int(never.sum()), LECTURE["never_inked"])
check("the largest weight on those squares, in any neuron", float(np.abs(W[:, never]).max()), 0.0)
r, c = divmod(int(freq.argmax()), 28)
check("the most-inked square (row, column, counting from 0)", [r, c], LECTURE["most_inked_square"][:2])
check("how often it is inked", float(freq.max()), LECTURE["most_inked_square"][2], tol=1e-4)

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.imshow(freq.reshape(28, 28), cmap=INKMAP)
rr, cc = np.where(never.reshape(28, 28))
ax.scatter(cc, rr, marker="x", s=14, color=INK, linewidths=0.8)
ax.set_title("how often each square is inked\n(x: never)", fontsize=10); ax.axis("off")
plt.show()

## 10. Ten templates (Plate XII)
Every neuron's weights, drawn the same way. Read them: the "1" is a bar of ink with rings on both sides (ink beside the
stroke means *not a 1*); the "0" is a ring of ink round a hole of rings. The biases say how eager each neuron is.

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(9, 4))
for d, ax in enumerate(axs.flat):
    template(ax, W[d], f'"{d}"   bias {minus(f"{b[d]:+.2f}")}')
plt.show()
for d in range(10):
    check(f'bias of "{d}"', b[d], LECTURE["weights"]["bias"][d], tol=0.01)
check("the most reluctant neuron (lowest bias)", int(b.argmin()), 8)
check("the most eager neuron (highest bias)", int(b.argmax()), 5)

## 11. Not the average digit (Plate XIII)
Why learn at all? Use the average 3 as the "3" template, and so on. Then compare what the learned weights look like:
the average digit, or the average digit *minus the average of all digits*? "Looks like" is the correlation of the two
pictures (1: identical up to brightness, 0: unrelated).

In [ ]:
avg = np.stack([xtr[tr_lab == d].mean(0) for d in range(10)])
check("average digits as templates: accuracy", accuracy(avg, np.zeros(10), xte, te_lab), LECTURE["mean_template_acc"], tol=1e-4)
corr = lambda a, c: float(np.corrcoef(a, c)[0, 1])
like_avg = [corr(W[d], avg[d]) for d in range(10)]
like_diff = [corr(W[d], avg[d] - xtr.mean(0)) for d in range(10)]
print("learned vs average digit:           ", np.round(like_avg, 2))
print("learned vs average minus all digits:", np.round(like_diff, 2))
check("looks like the average digit: lowest", min(like_avg), LECTURE["corr_with_average"][0], tol=0.01)
check("looks like the average digit: highest", max(like_avg), LECTURE["corr_with_average"][1], tol=0.01)
check("looks like the difference: lowest", min(like_diff), LECTURE["corr_with_difference"][0], tol=0.01)
check("looks like the difference: highest", max(like_diff), LECTURE["corr_with_difference"][1], tol=0.01)

fig, axs = plt.subplots(1, 3, figsize=(8, 2.9))
axs[0].imshow(avg[3].reshape(28, 28), cmap=INKMAP); axs[0].set_title("the average 3", fontsize=10); axs[0].axis("off")
template(axs[1], avg[3] - xtr.mean(0), "average 3 minus\nthe average of all")
template(axs[2], W[3], f"learned (looks like it: {like_diff[3]:.2f})")
plt.show()

## 12. Where it fails (Plate XIV)
Every test digit, by what it was and what the model said. The diagonal is right.

In [ ]:
said = scores(W, b, xte).argmax(1)
cm = np.zeros((10, 10), int)
np.add.at(cm, (te_lab, said), 1)
per = cm.diagonal() / cm.sum(1)
print("right, per digit:", " ".join(f"{d}: {v:.1%}" for d, v in enumerate(per)))
check('right for "0"s', per[0], LECTURE["per_digit_acc"][0], tol=0.003)
check('right for "5"s (the worst)', per[5], LECTURE["per_digit_acc"][5], tol=0.005)
check("the worst digit", int(per.argmin()), 5)
off = sorted(((cm[t, p], t, p) for t in range(10) for p in range(10) if t != p), reverse=True)[:3]
print("commonest mistakes:", ", ".join(f"{t} read as {p}: {n}" for n, t, p in off))
want = {(t, p): n for t, p, n in LECTURE["commonest_mistakes"]}
for n, t, p in off:
    check(f"{t} read as {p}", n, want.get((t, p), -99), tol=3)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(np.log1p(cm), cmap=INKMAP)
for t in range(10):
    for p in range(10):
        if cm[t, p] >= 20:
            ax.text(p, t, cm[t, p], ha="center", va="center", fontsize=6.5, color=PAPER if t == p else INK)
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xlabel("what the model said"); ax.set_ylabel("what it was")
ax.set_title("all 10,000 test digits", fontsize=10)
plt.show()

i = LECTURE["surest_mistake"]["index"]
s = scores(W, b, xte[i][None])[0]
print(f"the surest mistake, test #{i}: a {te_lab[i]} read as {s.argmax()} ({s.max():.2f} against {s[te_lab[i]]:.2f})")
check(f"test #{i} is a", int(te_lab[i]), LECTURE["surest_mistake"]["label"])
check(f"the model reads #{i} as", int(s.argmax()), LECTURE["surest_mistake"]["said"])

## 13. Templates don't move (Plate XV)
Each weight belongs to one fixed square. Move a digit two squares to the right (blank paper fills in from the left) and
its ink lands on weights meant for other strokes. Test digit #8570 is a clear 3.

In [ ]:
def shift(imgs, k):
    """Move every digit k squares to the right; blank paper comes in from the left."""
    out = np.zeros_like(imgs)
    out[:, :, k:] = imgs[:, :, :28 - k]
    return out

X8570 = te_img[8570:8571].astype(np.float32) / 255
for k, want in enumerate(LECTURE["digit8570"]):
    s = scores(W, b, shift(X8570, k).reshape(1, 784))[0]
    print(f"moved {k}: answer {s.argmax()}   score of 3 {s[3]:6.2f}   best {s.max():6.2f}")
    check(f"#8570 moved {k}: the answer", int(s.argmax()), want["pred"])
    check(f"#8570 moved {k}: the score of 3", s[3], want["score_true"], tol=0.03)

acc_k = [accuracy(W, b, shift(te_img.astype(np.float32) / 255, k).reshape(-1, 784), te_lab) for k in range(5)]
for k in range(5):
    check(f"whole test set, moved {k} squares", acc_k[k], LECTURE["shift_acc"][k], tol=0.001)

fig, axs = plt.subplots(1, 6, figsize=(10, 2), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 2.2]})
for k in range(5):
    axs[k].imshow(shift(X8570, k)[0], cmap=INKMAP); axs[k].axis("off")
    axs[k].set_title(f"moved {k}: says {scores(W, b, shift(X8570, k).reshape(1, 784))[0].argmax()}", fontsize=9)
axs[5].plot(range(5), acc_k, "o-", color=INK); axs[5].set_xlabel("squares moved"); axs[5].set_ylabel("test accuracy")
axs[5].set_ylim(0, 1)
plt.show()

A network whose detectors **slide** across the picture barely notices the move (98.7% at one square, 97.1% at two,
for the small sliding network of the next lecture). That is Lecture 6, and its notebook.

## Your turn
Five exercises. Each has a one-line hint; worked solutions, with checks, are at the end.

1. **Smoother templates.** Train again with weight decay 0.001 (`train(decay=1e-3)`). Does accuracy change? How alike
   are the new "3" weights and the old? *Hint: average the correlation over the ten neurons.*
2. **A different start.** Start from small random weights instead of zeros:
   `start=lambda g: 0.01 * torch.randn(10, 784, generator=g).numpy()` with `seed=7`. Same templates?
   *Hint: compare with `corr` as in §11.*
3. **The nearest average.** Score each digit by how close it is to each average digit: dot product minus half the
   average's squared length. How much of the gap to 92.1% does that close? *Hint: it is a bias of −½‖avg‖².*
4. **The surest 7 read as a 9.** Among 7s the model calls 9, which one has the biggest margin? *Hint: margin = score of
   9 minus score of 7.*
5. **Move digit #30.** Does our running example also fail when moved 1–4 squares? *Hint: reuse `shift`.*

In [ ]:
# Your turn 1
# TODO

In [ ]:
# Your turn 2
# TODO

In [ ]:
# Your turn 3
# TODO

In [ ]:
# Your turn 4
# TODO

In [ ]:
# Your turn 5
# TODO

## Solutions

In [ ]:
# 1. Smoother templates
Wd, bd, _ = train(decay=1e-3)
same = np.mean([corr(Wd[d], W[d]) for d in range(10)])
print(f"weight decay 0.001: {accuracy(Wd, bd, xte, te_lab):.2%} right; looks like the original: {same:.4f}")
check("weight decay 0.001: accuracy", accuracy(Wd, bd, xte, te_lab), LECTURE["wd_1e-3"]["acc"], tol=0.0005)
check("weight decay 0.001: alike", same, LECTURE["wd_1e-3"]["corr"], tol=0.0005)

In [ ]:
# 2. A different start
Wr, br, _ = train(seed=7, start=lambda g: 0.01 * torch.randn(10, 784, generator=g).numpy())
same = np.mean([corr(Wr[d], W[d]) for d in range(10)])
print(f"random start: {accuracy(Wr, br, xte, te_lab):.2%} right; looks like the original: {same:.4f}")
check("random start: accuracy", accuracy(Wr, br, xte, te_lab), LECTURE["random_start"]["acc"], tol=0.0005)
check("random start: alike", same, LECTURE["random_start"]["corr"], tol=0.0005)

In [ ]:
# 3. The nearest average
near = accuracy(avg, -0.5 * (avg ** 2).sum(1), xte, te_lab)
print(f"nearest average: {near:.1%}; plain average template: {LECTURE['mean_template_acc']:.1%}; learned: {LECTURE['test_acc']:.1%}")
check("nearest average: accuracy", near, LECTURE["nearest_mean_acc"], tol=1e-4)

In [ ]:
# 4. The surest 7 read as a 9
S_te = scores(W, b, xte)
sevens = np.where((te_lab == 7) & (S_te.argmax(1) == 9))[0]
j = int(sevens[np.argmax(S_te[sevens, 9] - S_te[sevens, 7])])
print(f"test #{j}: '9' scores {S_te[j, 9]:.2f}, '7' scores {S_te[j, 7]:.2f}")
check("the surest 7 read as a 9", j, LECTURE["surest_7_as_9"]["index"])
check("its score for 9", S_te[j, 9], LECTURE["surest_7_as_9"]["score9"], tol=0.02)
check("its score for 7", S_te[j, 7], LECTURE["surest_7_as_9"]["score7"], tol=0.02)
fig, ax = plt.subplots(figsize=(1.8, 1.8)); ax.imshow(te_img[j], cmap=INKMAP); ax.axis("off"); plt.show()

In [ ]:
# 5. Move digit #30
for k in range(5):
    s = scores(W, b, shift(te_img[30:31].astype(np.float32) / 255, k).reshape(1, 784))[0]
    check(f"#30 moved {k}: still a 3", int(s.argmax()), 3)
    check(f"#30 moved {k}: score of 3", s[3], LECTURE["digit30_shifted_score3"][k], tol=0.03)
print("#30 is a robust 3: a thick, centred digit. That is why the lecture needed #8570 to show the catch.")
print(f"notebook ran in {time.time() - T0:.0f} s")